In [13]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import pytest

In [14]:
from qrc_bloqade.encodings import amp_encode, angle_encode
from qrc_bloqade.hamiltonian import rydberg 
from qrc_bloqade.readouts import ZReadout
from qrc_bloqade.solver import BloqadeSolver 
from qrc_bloqade.utils import(train_split, normalize, get_predictor, print_readout) 


In [ ]:
def test_qrc_angle_encoding():
    n_sites = 4
    omega = 1.0
    V = 1.0
    time = 1.0

    # Create instances of the concrete classes
    hamiltonian = rydberg(n_sites, omega, V)
    encoder = angle_encode(n_sites)  # Use AngleEncoding
    readout = ZReadout(n_sites)
    predictor = get_predictor()
    scaler = MinMaxScaler(feature_range=(-1, 1))
    splitter = train_split

    # Generate synthetic data
    X, y = make_regression(n_samples=100, n_features=n_sites, noise=0.1, random_state=402)

    # Normalize data
    X_normalized, y_normalized, scaler_X, scaler_y = normalize(X, y)

    # Split data
    X_train, X_val, X_test, y_train, y_val, y_test = splitter(
        X_normalized, y_normalized, test_size=0.4, random_state=402
    )

    # --- QRC Workflow ---
    # Encode
    encoded_train = encoder.encode(X_train)
    encoded_test = encoder.encode(X_test)
    encoded_train = np.array(encoded_train)
    encoded_test = np.array(encoded_test)



    print(f"n_sites (qubits): {n_sites}")
    print(f"Expected state size: {2**n_sites}")
    print(f"Actual train state size: {len(encoded_train)}")
    print(f"Actual test state size: {len(encoded_test)}")
   # print(f"Encoded train size: {encoded_train.shape}")
    #print(f"Encoded test size: {encoded_test.shape}")
 
    # Evolve (Dynamics)
    solver = BloqadeSolver()
    evolved_train = [solver.simulate(
        sample, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100, n_qubits=n_sites
    ) for sample in encoded_train]
    evolved_test = [solver.simulate(
        sample, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100,  n_qubits=n_sites
    ) for sample in encoded_test]

    # Measure
    train_features = readout.measure(evolved_train)
    test_features = readout.measure(evolved_test)

    # Predict (Train and Test)
    predictor.fit(train_features, y_train)
    predictions = predictor.predict(test_features)

    # Inverse transform to get original scale
    predictions = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    # Evaluate
    mse = mean_squared_error(y_test, predictions)

    print(f"Test MSE (Angle Encoding): {mse}")
    assert mse >= 0  # Basic check

#

In [29]:
# Cell 3: Test function (Amplitude Encoding)
def test_qrc_amplitude_encoding():
    n_sites = 2  # Use only 2 qubits for easier amplitude encoding
    omega = 1.0
    V = 1.0
    time = 1.0

    # QRC components
    hamiltonian = rydberg(n_sites, omega, V)
    encoder = amp_encode(n_sites)  # Use AmplitudeEncoding
    readout = ZReadout(n_sites)
    predictor = get_predictor()
    scaler = MinMaxScaler(feature_range=(-1, 1)) # Needed just for making y in original scale
    splitter = train_split

    # --- Data Preparation (for amplitude encoding) ---
    # Generate synthetic data with the correct number of features (2^n_sites)
    X = np.random.rand(100, 2**n_sites)  # Correct number of features for amplitude encoding
    y = np.random.rand(100)

    # Normalize data for Amplitude Encoding (sum of squares = 1)
    X_normalized = []
    for row in X:
        norm = np.linalg.norm(row)
        normalized_row = row / norm
        X_normalized.append(normalized_row)
    X_normalized = np.array(X_normalized)
    
    # Normalize y values 
    y_normalized, scaler_y = normalize(X_normalized,y)[:2]  

    # No padding needed if X already has the correct number of features
    X_train, X_val, X_test, y_train, y_val, y_test = splitter(
    X_normalized, y_normalized, test_size=0.4, random_state=402)    

    # --- QRC Workflow ---
    # Encode
    encoded_train = encoder.encode(X_train)
    encoded_test = encoder.encode(X_test)
    print(f"n_sites (qubits): {n_sites}")
    print(f"Expected state size: {2**n_sites}")
    print(f"Actual train state size: {len(encoded_train)}")
    print(f"Actual test state size: {len(encoded_test)}")
    print(f"Encoded train size: {encoded_train.shape}")
    print(f"Encoded test size: {encoded_test.shape}")

    # Evolve
    solver = BloqadeSolver()
    evolved_train = solver.simulate(
        encoded_train, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100, n_qubits=n_sites
    )
    evolved_test = solver.simulate(
        encoded_test, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100, n_qubits=n_sites
    )

    # Measure
    train_features = readout.measure(evolved_train)
    test_features = readout.measure(evolved_test)

    # Predict
    predictor.fit(train_features, y_train)
    predictions = predictor.predict(test_features)

    # Inverse transform to get original scale.
    predictions = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    # Evaluate
    mse = mean_squared_error(y_test, predictions)
    print(f"Test MSE (Amplitude Encoding): {mse}")
    assert mse >= 0

# Cell 4: Main execution block


In [30]:
if __name__ == "__main__":
    print("Running Angle Encoding Test...")
    test_qrc_angle_encoding()
    print("\nRunning Amplitude Encoding Test...")
    test_qrc_amplitude_encoding()

Running Angle Encoding Test...
n_sites (qubits): 4
Expected state size: 16
Actual train state size: 60
Actual test state size: 20
Encoded train size: (60, 16)
Encoded test size: (20, 16)
Initial state size: 16


AttributeError: module 'bloqade.emulate.ir.emulator' has no attribute 'run'